# WiFi CSI Human Activity Recognition — CNN-GRU with AttentionBinary classification of 2-second WiFi CSI amplitude windows (52 x 200):`action` (person moving) vs `noaction` (person present but still).**Capture.** USRP N210 passively sniffing an 802.11 link, channel 11 (2.462 GHz),20 MHz, single antenna, 52 OFDM subcarriers. Amplitude `|H|` only — the USRP is notfrequency- or time-synchronised to the transmitter, so per-packet phase carries anunknown rotation and is not usable.**Evaluation.** Leave-One-Subject-Out: train on one subject, test on the other.Subjects are anonymised as `S01`, `S02`, `S03`.**Inputs.** Windows produced by `scripts/segment_windows.py` in`../data/pilot/windows/{action,noaction}/*.npy`, each of shape `(52, 200)`.

In [ ]:
import os
import re
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"torch {torch.__version__}")

PILOT   = "../data/pilot"
WINDOWS = f"{PILOT}/windows"
MODELS  = "../models"
os.makedirs(MODELS, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

## 1. Raw capturesEach decoded 802.11 frame yields one CSV row: a timestamp plus 52 complex channelestimates, one per OFDM subcarrier. Duration and packet rate are derived from thetimestamps, not from row counts, because the sniffed link emits packets irregularly.

In [ ]:
csv_files = sorted(glob.glob(f"{PILOT}/P01_*.csv"))
print(f"{len(csv_files)} capture files\n")

def short_name(path):
    return re.sub(r"^P01_|_\d{8}|\.csv$", "", os.path.basename(path))

def load_capture(path):
    '''CSV -> (time_s, amplitude[n_packets, 52], n_dropped_rows).

    The final row of each capture is truncated because the flowgraph was stopped
    mid-write, so incomplete rows are dropped explicitly.
    '''
    df = pd.read_csv(path)
    n_sc = sum(1 for c in df.columns if c.endswith("_Re"))
    cols = [f"SC{i}_Re" for i in range(n_sc)] + [f"SC{i}_Im" for i in range(n_sc)]
    before = len(df)
    df = df.dropna(subset=cols + ["Timestamp_us"])
    dropped = before - len(df)

    t  = df["Timestamp_us"].to_numpy(np.int64)
    re_ = df[[f"SC{i}_Re" for i in range(n_sc)]].to_numpy(float)
    im_ = df[[f"SC{i}_Im" for i in range(n_sc)]].to_numpy(float)
    amp = np.hypot(re_, im_)                       # |H| : the sensing signal

    o = np.argsort(t)
    return (t[o] - t[o][0]) / 1e6, amp[o], dropped

rows = []
for f in csv_files:
    t, amp, dropped = load_capture(f)
    rows.append({
        "file": short_name(f),
        "class": "action" if "_action_" in os.path.basename(f) else "noaction",
        "packets": len(t),
        "duration_s": round(t[-1], 1),
        "rate_Hz": round((len(t) - 1) / t[-1], 1),
        "truncated_rows": dropped,
    })

inventory = pd.DataFrame(rows)
print(inventory.to_string(index=False))
print(f"\nMean packet rate: {inventory.rate_Hz.mean():.1f} Hz   "
      f"(min {inventory.rate_Hz.min()}, max {inventory.rate_Hz.max()})")

In [ ]:
# Amplitude over a full take, and packet-rate stability
ACT_EXAMPLE   = "P01_action_S03_20260717_walk.csv"
NOACT_EXAMPLE = "P01_noaction_S03_20260717_stand.csv"

fig, axes = plt.subplots(2, 2, figsize=(14, 6.5))
for col, (fn, title) in enumerate([(ACT_EXAMPLE, "action - walk"),
                                   (NOACT_EXAMPLE, "noaction - stand")]):
    t, amp, _ = load_capture(f"{PILOT}/{fn}")

    ax = axes[0, col]
    for sc in (0, 13, 26, 39, 51):
        ax.plot(t, amp[:, sc], lw=0.45, alpha=0.85, label=f"SC{sc}")
    ax.set_title(f"{title}: |H| over the whole {t[-1]:.0f} s take")
    ax.set_xlabel("time (s)"); ax.set_ylabel("|H|")
    ax.legend(fontsize=7, ncol=5, loc="upper right")

    ax = axes[1, col]
    per_sec = np.bincount(t.astype(int))[:-1]
    ax.bar(np.arange(len(per_sec)), per_sec, width=1.0, alpha=0.85)
    ax.set_title(f"{title}: packets per second (mean {len(t)/t[-1]:.0f} Hz)")
    ax.set_xlabel("second"); ax.set_ylabel("packets/s")

plt.tight_layout(); plt.show()

In [ ]:
# Subcarrier x time view: the 2D structure the model operates on
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (fn, title) in zip(axes, [(ACT_EXAMPLE, "action - walk"),
                                  (NOACT_EXAMPLE, "noaction - stand")]):
    t, amp, _ = load_capture(f"{PILOT}/{fn}")
    m = t <= 20                                        # first 20 s
    d = amp[m] - amp[m].mean(axis=0, keepdims=True)    # remove static offset
    v = np.percentile(np.abs(d), 99)
    im = ax.imshow(d.T, aspect="auto", cmap="RdBu_r", vmin=-v, vmax=v,
                   extent=[0, t[m][-1], 52, 0])
    ax.set_title(f"{title}: all 52 subcarriers (mean-removed)")
    ax.set_xlabel("time (s)"); ax.set_ylabel("subcarrier index"); ax.grid(False)
    plt.colorbar(im, ax=ax, label="dH")
plt.tight_layout(); plt.show()

## 2. Windows and dataset composition`scripts/segment_windows.py` turns each capture into 2-second windows: it computes`|H|` per packet, resamples a 2 s span onto a uniform time grid by linearinterpolation using the real timestamps (giving exactly 52 x 200), and skips thewindow if fewer than 50% of the expected packets are present.

In [ ]:
# The model's actual input tensor, one window per class
act_files   = sorted(glob.glob(f"{WINDOWS}/action/*.npy"))
noact_files = sorted(glob.glob(f"{WINDOWS}/noaction/*.npy"))

if not act_files or not noact_files:
    raise FileNotFoundError(
        f"No segmented windows under {WINDOWS}. They are generated output and are\n"
        "not committed - regenerate them from the raw captures first:\n\n"
        "  for f in data/pilot/P01_*.csv; do\n"
        "    b=$(basename \"$f\" .csv)\n"
        "    python3 scripts/segment_windows.py \"$f\" \\\n"
        "        --action  \"$(echo \"$b\" | cut -d_ -f2)\" \\\n"
        "        --subject \"$(echo \"$b\" | cut -d_ -f3)\" \\\n"
        "        --out-dir data/pilot/windows\n"
        "  done\n\nSee data/README.md."
    )

print(f"{len(act_files)} action windows, {len(noact_files)} noaction windows")

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for ax, (f, title) in zip(axes, [(act_files[0], "action"), (noact_files[0], "noaction")]):
    w = np.load(f)
    d = w - w.mean(axis=1, keepdims=True)          # exactly what the Dataset does
    v = np.percentile(np.abs(d), 99)
    im = ax.imshow(d, aspect="auto", cmap="RdBu_r", vmin=-v, vmax=v, extent=[0, 2, 52, 0])
    ax.set_title(f"{title}: model input (52 x 200), detrended")
    ax.set_xlabel("time (s)"); ax.set_ylabel("subcarrier"); ax.grid(False)
    plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

In [ ]:
recs = []
for cls in ("action", "noaction"):
    for p in sorted(glob.glob(f"{WINDOWS}/{cls}/*.npy")):
        b = os.path.basename(p)
        recs.append({
            "class": cls,
            "subject": re.search(r"_(S\d+)_", b).group(1),
            "take": next(k for k in ("walk", "mixed", "stand", "sit") if k in b),
        })
ds = pd.DataFrame(recs)

print("Windows per class:");    print(ds["class"].value_counts().to_string())
print("\nWindows per subject:"); print(ds["subject"].value_counts().to_string())
print("\nSubject x take:");      print(pd.crosstab(ds["subject"], ds["take"]).to_string())
print(f"\nTotal windows: {len(ds)}   class balance: "
      f"{100*(ds['class']=='action').mean():.1f}% action")

fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.2))
ds["class"].value_counts().plot.bar(ax=axes[0], color=["#c0392b", "#2471a3"], rot=0)
axes[0].set_title("windows per class"); axes[0].set_ylabel("count")
pd.crosstab(ds["subject"], ds["class"]).plot.bar(ax=axes[1], rot=0,
                                                 color=["#c0392b", "#2471a3"])
axes[1].set_title("per subject"); axes[1].legend(fontsize=7)
ds["take"].value_counts().plot.bar(ax=axes[2], rot=0, color="#7f8c8d")
axes[2].set_title("per take type")
plt.tight_layout(); plt.show()

## 3. PreprocessingTwo per-window operations:1. **Detrend** — subtract each subcarrier's temporal mean. `|H|` has a large static   component set by room geometry, antenna gain and distance that carries no motion   information; removing it leaves only the fluctuation.2. **Scale** — divide by the per-subcarrier standard deviation computed on the   **training split only**, so no test statistics leak into training. Subcarriers have   different baseline variances, and without this a few loud ones dominate the gradient.

In [ ]:
class CsiAmplitudeDataset(Dataset):
    def __init__(self, items, classes, train_std):
        self.items = items                 # list of (path, class_idx)
        self.classes = classes
        self.train_std = train_std

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, y = self.items[i]
        window = np.load(path).astype(np.float32)
        window = window - window.mean(axis=1, keepdims=True)   # detrend
        window = window / self.train_std                       # scale
        return torch.tensor(window), torch.tensor(y)


def scan_dataset(root):
    items = []
    classes = sorted(d for d in os.listdir(root)
                     if os.path.isdir(os.path.join(root, d)))
    for ci, c in enumerate(classes):
        for f in sorted(os.listdir(os.path.join(root, c))):
            if f.endswith(".npy"):
                items.append((os.path.join(root, c, f), ci))
    return items, classes

## 4. Model — CNN -> BiGRU -> Attention| Stage | Layer | Role ||---|---|---|| Spatial | `Conv1d(52 -> 64, k=5)` + BN + ReLU + MaxPool | The 52 subcarriers are the input *channels*, so each filter sees all of them at every time step and can learn cross-subcarrier patterns. Time 200 -> 100. || | `Conv1d(64 -> 128, k=5)` + BN + ReLU + MaxPool | Composes motion primitives into longer patterns. Time 100 -> 50. || Temporal | `GRU(128 -> 128, 2 layers, bidirectional)` | Carries state across the full 2 s; an activity is an ordered sequence. Bidirectional because windows are classified offline. || Pooling | `Linear(256 -> 1)` + softmax over time | Attention. Motion may occupy only part of the window, so a learned weighted average beats mean-pooling. || Head | `Linear(256 -> 64)` -> ReLU -> `Dropout(0.5)` -> `Linear(64 -> 2)` | Small head with heavy dropout for a small dataset. |

In [ ]:
class SpatialTemporalAttentionModel(nn.Module):
    def __init__(self, num_classes=2, num_subcarriers=52, cnn_channels=64, gru_hidden=128):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Conv1d(num_subcarriers, cnn_channels, kernel_size=5, padding=2),
            nn.BatchNorm1d(cnn_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2),                       # time 200 -> 100

            nn.Conv1d(cnn_channels, cnn_channels * 2, kernel_size=5, padding=2),
            nn.BatchNorm1d(cnn_channels * 2),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2),                       # time 100 -> 50
        )

        self.gru = nn.GRU(
            input_size=cnn_channels * 2,
            hidden_size=gru_hidden,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
        )

        self.attention_query = nn.Linear(gru_hidden * 2, 1)

        self.classifier = nn.Sequential(
            nn.Linear(gru_hidden * 2, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        features = self.feature_extractor(x)      # (batch, 128, 50)
        features = features.transpose(1, 2)       # (batch, 50, 128)

        out, _ = self.gru(features)               # (batch, 50, 256)

        attn_weights = torch.softmax(self.attention_query(out), dim=1)   # (batch, 50, 1)
        context = torch.sum(out * attn_weights, dim=1)                   # (batch, 256)

        return self.classifier(context)


print(f"{sum(p.numel() for p in SpatialTemporalAttentionModel().parameters()):,} parameters")

## 5. Training and Leave-One-Subject-Out evaluationAdam (lr 1e-3), exponential LR decay (gamma 0.85), cross-entropy, 30 epochs, batch 32.The split is **by subject, never random**: consecutive windows from one take are highlycorrelated, so a random split puts near-duplicates on both sides and reports ameaningless number. Model selection uses a validation set carved out of the *training*subject's windows, never the test subject.

In [ ]:
def evaluate(model, loader, device, n_classes):
    model.eval()
    correct = total = 0
    conf = np.zeros((n_classes, n_classes), int)
    all_pred, all_true = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            pred = model(x).argmax(1).cpu()
            correct += (pred == y).sum().item()
            total += len(y)
            all_pred.extend(pred.tolist()); all_true.extend(y.tolist())
            for yi, pi in zip(y.tolist(), pred.tolist()):
                conf[yi][pi] += 1
    return correct / max(total, 1), conf, np.array(all_true), np.array(all_pred)

In [ ]:
def run_loso_split(dataset_dir, test_subject, epochs=30, batch_size=32, lr=1e-3, verbose=True):
    items, classes = scan_dataset(dataset_dir)
    print("=" * 60)
    print(f"LOSO split: testing on subject -> {test_subject}")
    print("=" * 60)

    key = f"_{test_subject.lower()}_"
    test      = [it for it in items if key in os.path.basename(it[0]).lower()]
    train_all = [it for it in items if key not in os.path.basename(it[0]).lower()]

    # Validation split drawn from the training subject only
    wids = sorted({os.path.basename(p_) for p_, _ in train_all})
    rng = np.random.default_rng(0)
    val_wids = set(rng.choice(wids, max(1, int(len(wids) * 0.15)), replace=False))
    train = [it for it in train_all if os.path.basename(it[0]) not in val_wids]
    val   = [it for it in train_all if os.path.basename(it[0]) in val_wids]
    print(f"Windows: train={len(train)}, val={len(val)}, test={len(test)}")

    # Normalisation statistics from the TRAINING split only
    train_vals = [np.load(p) - np.load(p).mean(axis=1, keepdims=True) for p, _ in train]
    train_std = (np.concatenate(train_vals, axis=1).std(axis=1, keepdims=True)
                 + 1e-8).astype(np.float32)

    train_loader = DataLoader(CsiAmplitudeDataset(train, classes, train_std),
                              batch_size=batch_size, shuffle=True,  num_workers=2)
    val_loader   = DataLoader(CsiAmplitudeDataset(val, classes, train_std),
                              batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader  = DataLoader(CsiAmplitudeDataset(test, classes, train_std),
                              batch_size=batch_size, shuffle=False, num_workers=2)

    model   = SpatialTemporalAttentionModel(len(classes)).to(device)
    opt     = torch.optim.Adam(model.parameters(), lr=lr)
    sched   = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=0.85)
    loss_fn = nn.CrossEntropyLoss()

    best_val = 0.0
    model_path = os.path.join(MODELS, f"hybrid_model_test_{test_subject}.pt")
    history = {"loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        model.train()
        tot_loss = correct = seen = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            out = model(x)
            loss = loss_fn(out, y)
            loss.backward()
            opt.step()
            tot_loss += loss.item() * len(y)
            correct  += (out.argmax(1) == y).sum().item()
            seen     += len(y)
        sched.step()

        val_acc, _, _, _ = evaluate(model, val_loader, device, len(classes))
        history["loss"].append(tot_loss / seen)
        history["train_acc"].append(correct / seen)
        history["val_acc"].append(val_acc)

        if verbose and (epoch % 5 == 0 or epoch == epochs):
            print(f"Epoch {epoch:2d} | loss {tot_loss/seen:.4f} | "
                  f"train acc {correct/seen:.3f} | val acc {val_acc:.3f}")

        if val_acc >= best_val:                    # selection on VAL, never on TEST
            best_val = val_acc
            torch.save({"model": model.state_dict(), "classes": classes}, model_path)

    model.load_state_dict(torch.load(model_path, map_location=device,
                                     weights_only=True)["model"])
    test_acc, conf, y_true, y_pred = evaluate(model, test_loader, device, len(classes))
    print(f"\nFinal test per-window accuracy: {test_acc:.3%}")
    print("\nConfusion matrix (rows=true, cols=pred):")
    w = max(len(c) for c in classes) + 2
    print(" " * w + "".join(c[:6].rjust(8) for c in classes))
    for i, c in enumerate(classes):
        print(c.ljust(w) + "".join(str(conf[i][j]).rjust(8) for j in range(len(classes))))
    print()

    return {"acc": test_acc, "conf": conf, "history": history, "classes": classes,
            "model": model, "test_items": test, "train_std": train_std,
            "y_true": y_true, "y_pred": y_pred}

In [ ]:
# Subjects that have segmented windows available
subjects = sorted(ds["subject"].unique())
print(f"Running LOSO over: {subjects}\n")

results = {s: run_loso_split(WINDOWS, test_subject=s) for s in subjects}

print("=" * 60)
for s, r in results.items():
    print(f"test={s} : {r['acc']:.3%}")
print(f"Worst fold: {min(r['acc'] for r in results.values()):.3%}")
print("=" * 60)

## 6. Results

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(6.5 * len(results), 3.6), squeeze=False)
for ax, (name, r) in zip(axes[0], results.items()):
    h = r["history"]
    ep = np.arange(1, len(h["loss"]) + 1)
    ax2 = ax.twinx()
    ax2.plot(ep, h["loss"], c="grey", ls=":", lw=1.4, label="train loss")
    ax2.set_ylabel("loss", color="grey")
    ax.plot(ep, h["train_acc"], c="#c0392b", lw=1.7, label="train acc")
    ax.plot(ep, h["val_acc"],   c="#2471a3", lw=1.7, label="val acc")
    ax.axhline(r["acc"], c="green", ls="--", lw=1.3, label=f"final test {r['acc']:.3f}")
    ax.set_title(f"test subject = {name}", fontsize=10)
    ax.set_xlabel("epoch"); ax.set_ylabel("accuracy"); ax.set_ylim(0.4, 1.03)
    ax.legend(fontsize=7.5, loc="lower right")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(4.5 * len(results), 3.8), squeeze=False)
for ax, (name, r) in zip(axes[0], results.items()):
    cm, cls = r["conf"], r["classes"]
    ax.imshow(cm / cm.sum(axis=1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
    for i in range(len(cls)):
        for j in range(len(cls)):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=14,
                    color="white" if i == j else "black")
    ax.set_xticks(range(len(cls))); ax.set_xticklabels(cls)
    ax.set_yticks(range(len(cls))); ax.set_yticklabels(cls)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.grid(False)
    ax.set_title(f"test = {name}  ({r['acc']:.2%}, n={cm.sum()})", fontsize=10)
plt.tight_layout(); plt.show()

for name, r in results.items():
    cm = r["conf"]
    tp, fn_ = cm[0, 0], cm[0, 1]
    fp = cm[1, 0]
    prec = tp / max(tp + fp, 1)
    rec  = tp / max(tp + fn_, 1)
    print(f"test={name} | action precision {prec:.3f}  recall {rec:.3f}  "
          f"F1 {2*prec*rec/max(prec+rec, 1e-9):.3f}")